In [ ]:
from utils import *
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import json

In [ ]:
def adaptive_ratio_v1_0_loader():
    results_dir = Path("../results/adaptive_ratio_v1_0")
    scalars = TBScalars(".cache/adaptive_ratio_v1_0")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_v1_0_loader():
    results_dir = Path("../results/adaptive_wm_ratio_v1_0")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v1_0")

    res_df = []
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})
    res_df = pd.DataFrame.from_records(res_df)
    res_df

    return res_df, scalars


def adaptive_wm_ratio_v1_1_loader():
    res_df = []

    results_dir = Path("../results/adaptive_wm_ratio_v1_1")
    scalars = TBScalars(".cache/adaptive_wm_ratio_v1_1")
    for test in tqdm([*results_dir.iterdir()]):
        params = test.name.split("-")
        test_r = {}
        test_r["env"] = params[0]
        types = {"rl_ratio": int, "seed": int}
        for (name, typ), value in zip(types.items(), params[1:]):
            test_r[name] = typ(value.removeprefix(f"{name}="))
        df = scalars.read(test)
        scores = df[df["tag"] == "val/mean_ep_ret"]["value"]
        test_r["score"] = scores.iloc[-1]
        res_df.append({"path": test, **test_r})

    res_df = pd.DataFrame.from_records(res_df)

    return res_df, scalars


res_v10_df, scalars_v10 = adaptive_ratio_v1_0_loader()
res_wm_v10_df, scalars_wm_v10 = adaptive_wm_ratio_v1_0_loader()
res_wm_v11_df, scalars_wm_v11 = adaptive_wm_ratio_v1_1_loader()
res_dfs = {"v1.0": res_v10_df, "v1.0-wm": res_wm_v10_df, "v1.1-wm": res_wm_v11_df}

In [ ]:
envs = res_wm_v11_df["env"].unique()

fig = make_subplots(
    rows=3,
    row_titles=[*envs],
    cols=1,
)

for row, env in enumerate(envs, 1):
    dfs = []
    for name in res_dfs:
        df = res_dfs[name]
        df = df[df["env"] == env].copy()
        if "rl_ratio" in df.columns:
            df["tag"] = [f"{name}({r['rl_ratio']})" for _, r in df.iterrows()]
        else:
            df["tag"] = name
        dfs.append(df)

    df = pd.concat(dfs, axis=0)
    for tag in sorted(df["tag"].unique()):
        fig.add_trace(
            go.Box(
                y=df[df["tag"] == tag]["score"],
                name=tag,
                boxmean="sd",
                boxpoints="all",
            ),
            row=row,
            col=1,
        )

fig.update_layout(width=1024, height=1500)
fig